# Customer Retention Analysis

This notebook analyzes cleaned member and transaction data to measure customer retention, revenue retention, customer longevity, and segment performance.

**Churn definition:** a customer active in the prior month but inactive in the current month. Revenue at risk is represented by the prior-month revenue associated with those customers.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

DATA_DIR = Path(".")
MEMBER_FILE = DATA_DIR / "DA_Project_Member_Data.csv"
REVENUE_FILE = DATA_DIR / "DA_Project_Revenue_Data.csv"

## 1. Load and validate data

The original assessment provided two cleaned files joined by `member_ID`. The source files are intentionally not included in this public portfolio.

In [ ]:
required_member_columns = {
    "member_ID", "Employees", "Billing State/Province",
    "Industry", "Cancellation Date"
}
required_revenue_columns = {
    "member_ID", "Invoice: Invoice Date", "Invoice Item: Charge Amount"
}

if not MEMBER_FILE.exists() or not REVENUE_FILE.exists():
    raise FileNotFoundError(
        "Place DA_Project_Member_Data.csv and DA_Project_Revenue_Data.csv "
        "in the project directory before running this notebook."
    )

members_raw = pd.read_csv(MEMBER_FILE)
revenue_raw = pd.read_csv(REVENUE_FILE)

missing_member = required_member_columns - set(members_raw.columns)
missing_revenue = required_revenue_columns - set(revenue_raw.columns)
if missing_member or missing_revenue:
    raise ValueError(
        f"Missing member fields: {sorted(missing_member)}; "
        f"missing revenue fields: {sorted(missing_revenue)}"
    )

print(f"Members: {len(members_raw):,}")
print(f"Transactions: {len(revenue_raw):,}")

In [ ]:
revenue = revenue_raw.copy()
revenue["Invoice: Invoice Date"] = pd.to_datetime(
    revenue["Invoice: Invoice Date"], errors="coerce"
)
revenue["Invoice Item: Charge Amount"] = pd.to_numeric(
    revenue["Invoice Item: Charge Amount"], errors="coerce"
)
revenue = revenue.dropna(
    subset=["member_ID", "Invoice: Invoice Date", "Invoice Item: Charge Amount"]
)
revenue["InvoiceMonth"] = revenue["Invoice: Invoice Date"].dt.to_period("M")

revenue.head()

## 2. Monthly business trends

In [ ]:
monthly = (
    revenue.groupby("InvoiceMonth")
    .agg(
        Revenue=("Invoice Item: Charge Amount", "sum"),
        TotalOrders=("member_ID", "size"),
        UniqueMembers=("member_ID", "nunique"),
    )
    .reset_index()
)
monthly

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(monthly["InvoiceMonth"].astype(str), monthly["Revenue"], label="Revenue")
ax2.plot(monthly["InvoiceMonth"].astype(str), monthly["UniqueMembers"], label="Unique members")

ax1.set_title("Monthly Revenue and Active Customers")
ax1.set_xlabel("Month")
ax1.set_ylabel("Revenue")
ax2.set_ylabel("Unique members")
ax1.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

## 3. New and existing customer activity

In [ ]:
first_purchase = (
    revenue.groupby("member_ID", as_index=False)["InvoiceMonth"]
    .min()
    .rename(columns={"InvoiceMonth": "MembershipStart"})
)

purchases = revenue.merge(first_purchase, on="member_ID", how="left")
purchases["CustomerType"] = np.where(
    purchases["InvoiceMonth"].eq(purchases["MembershipStart"]),
    "New",
    "Existing",
)

customer_type_monthly = (
    purchases.groupby(["InvoiceMonth", "CustomerType"])
    .agg(
        Revenue=("Invoice Item: Charge Amount", "sum"),
        TotalOrders=("member_ID", "size"),
        UniqueMembers=("member_ID", "nunique"),
    )
    .reset_index()
)
customer_type_monthly.head()

## 4. Customer and revenue retention

Rates use the prior month's active customer or revenue base as the denominator. This avoids understating churn when the current month includes substantial new-customer growth.

In [ ]:
activity = (
    revenue.assign(active=1)
    .pivot_table(index="member_ID", columns="InvoiceMonth", values="active", aggfunc="max", fill_value=0)
    .astype(int)
    .sort_index(axis=1)
)

revenue_matrix = (
    revenue.pivot_table(
        index="member_ID",
        columns="InvoiceMonth",
        values="Invoice Item: Charge Amount",
        aggfunc="sum",
        fill_value=0,
    )
    .sort_index(axis=1)
)

retention_rows = []
months = list(activity.columns)
for prior_month, current_month in zip(months[:-1], months[1:]):
    prior_active = activity[prior_month].eq(1)
    current_active = activity[current_month].eq(1)

    prior_users = int(prior_active.sum())
    retained_users = int((prior_active & current_active).sum())
    churned_users = int((prior_active & ~current_active).sum())
    new_users = int((~prior_active & current_active).sum())

    prior_revenue = float(revenue_matrix.loc[prior_active, prior_month].sum())
    retained_revenue = float(revenue_matrix.loc[prior_active & current_active, current_month].sum())
    churn_revenue = float(revenue_matrix.loc[prior_active & ~current_active, prior_month].sum())

    retention_rows.append({
        "InvoiceMonth": current_month,
        "PriorActiveUsers": prior_users,
        "RetainedUsers": retained_users,
        "ChurnedUsers": churned_users,
        "NewUsers": new_users,
        "RetentionRate": retained_users / prior_users if prior_users else np.nan,
        "ChurnRate": churned_users / prior_users if prior_users else np.nan,
        "PriorMonthRevenue": prior_revenue,
        "RetainedCurrentRevenue": retained_revenue,
        "RevenueAtRisk": churn_revenue,
        "RevenueAtRiskRate": churn_revenue / prior_revenue if prior_revenue else np.nan,
    })

retention = pd.DataFrame(retention_rows)
retention

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(retention["InvoiceMonth"].astype(str), retention["ChurnRate"], label="Customer churn rate")
ax.plot(retention["InvoiceMonth"].astype(str), retention["RevenueAtRiskRate"], label="Revenue-at-risk rate")
ax.set_title("Monthly Customer and Revenue Churn")
ax.set_xlabel("Month")
ax.set_ylabel("Rate")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
plt.show()

## 5. Customer longevity and behavioral features

In [ ]:
customer_summary = (
    revenue.groupby("member_ID")
    .agg(
        MonthsActive=("InvoiceMonth", "nunique"),
        TotalSpent=("Invoice Item: Charge Amount", "sum"),
        Purchases=("Invoice Item: Charge Amount", "size"),
        FirstPurchase=("Invoice: Invoice Date", "min"),
        LastPurchase=("Invoice: Invoice Date", "max"),
    )
    .reset_index()
)
customer_summary["AverageOrderValue"] = (
    customer_summary["TotalSpent"] / customer_summary["Purchases"]
)

members = members_raw.merge(customer_summary, on="member_ID", how="left")
members["Cancellation Date"] = pd.to_datetime(members["Cancellation Date"], errors="coerce")
members["Canceled"] = members["Cancellation Date"].notna().astype(int)

print(f"Mean active months: {members['MonthsActive'].mean():.1f}")
print(f"Median active months: {members['MonthsActive'].median():.1f}")
print(
    "Median active months among canceled customers: "
    f"{members.loc[members['Canceled'].eq(1), 'MonthsActive'].median():.1f}"
)

In [ ]:
size_bins = [0, 25, 100, np.inf]
size_labels = ["1-25", "26-100", "101+"]
members["CompanySize"] = pd.cut(
    pd.to_numeric(members["Employees"], errors="coerce"),
    bins=size_bins,
    labels=size_labels,
    include_lowest=True,
)

state_to_region = {
    **{s: "Northeast" for s in ["Maine", "New Hampshire", "Massachusetts", "Connecticut", "Vermont", "Rhode Island", "New York", "Pennsylvania", "New Jersey"]},
    **{s: "Midwest" for s in ["Wisconsin", "Michigan", "Illinois", "Indiana", "Ohio", "Minnesota", "North Dakota", "South Dakota", "Nebraska", "Kansas", "Iowa", "Missouri"]},
    **{s: "South" for s in ["Maryland", "Virginia", "West Virginia", "North Carolina", "South Carolina", "Georgia", "Florida", "District of Columbia", "Mississippi", "Tennessee", "Kentucky", "Alabama", "Louisiana", "Arkansas", "Texas", "Oklahoma"]},
    **{s: "West" for s in ["Colorado", "New Mexico", "Arizona", "Utah", "Montana", "Nevada", "Idaho", "California", "Washington", "Oregon", "Hawaii", "Alaska"]},
}
members["Region"] = members["Billing State/Province"].map(state_to_region)

members.head()

## 6. Segment performance

In [ ]:
def segment_summary(data: pd.DataFrame, segment: str, minimum_members: int = 5) -> pd.DataFrame:
    summary = (
        data.groupby(segment, observed=True)
        .agg(
            Members=("member_ID", "nunique"),
            MeanActiveMonths=("MonthsActive", "mean"),
            CancellationRate=("Canceled", "mean"),
            MeanCustomerSpend=("TotalSpent", "mean"),
            MeanOrderValue=("AverageOrderValue", "mean"),
        )
        .query("Members >= @minimum_members")
        .sort_values(["MeanActiveMonths", "CancellationRate"], ascending=[False, True])
    )
    return summary

industry_summary = segment_summary(members, "Industry")
size_summary = segment_summary(members, "CompanySize")
region_summary = segment_summary(members, "Region")

industry_summary.head(10)

In [ ]:
size_summary

In [ ]:
region_summary

## 7. Interpretation framework

When reviewing the results, prioritize segments that combine:

1. Sufficient customer volume
2. Longer active lifetimes
3. Lower cancellation rates
4. Higher customer spend or order value

Small segments can produce unstable averages, so minimum group-size thresholds should be applied before making targeting decisions. Churn spikes should also be evaluated alongside acquisition campaigns, seasonality, pricing changes, and service events before assigning a cause.

## Limitations

- Month-over-month inactivity is an operational churn definition; it may classify temporarily inactive customers as churned.
- The analysis is descriptive and does not establish causal relationships.
- Revenue at risk uses prior-month revenue and should not be interpreted as a forecast of permanent loss.
- Segment comparisons may reflect differences in tenure, acquisition channel, or customer mix.
- The original assessment data is not included, so results cannot be reproduced from this repository alone.